# Income & the Prenatal Care Access Gap — Deep Dive
### The Lily Foundation & Advocates

Follow-up to `Vital_Signs_Cleaning_and_EDA.ipynb`. One question, answered in order:
**how does income drive the early-prenatal-care gap, and where does that gap concentrate
and persist?**

This notebook **does not re-clean the raw workbook** and does not recompute the Service Need
Score or the four-dimension profile. It consumes the outputs the original notebook wrote to
`tableau/Baltimore_Vital_Signs_Tableau.xlsx`:

| Sheet read | Used for |
|---|---|
| `Historical_Trends` | every indicator value, all years, already cleaned (`norm_name`-matched, `--` → null, city row held out as `CityValue`) |
| `Service_Need_Profile` | each CSA's `Tier` and `Composite` (the four-dimension profile, 2023) |

The raw workbook is opened **for column headers only** in section 1, to confirm the indicator
search covers all 14 year-sheets — including any spelling that was dropped from
`Historical_Trends` because it was empty.

As in the original: **no charts here** — visuals are built in Tableau. All correlations are
**ecological, across 55 areas**, not individual-level effects.

## 0. Setup

In [1]:
import os, re, glob, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

# ASSUMPTION: same relative layout as the original notebook -- run from the project root,
# raw workbook in raw/, Tableau workbook in tableau/.
OUT      = "tableau"
TABLEAU  = f"{OUT}/Baltimore_Vital_Signs_Tableau.xlsx"
XLSX     = "raw/Vital Signs 2010-2023.xlsx"          # headers only, section 1
if not os.path.exists(XLSX):
    hits = glob.glob("raw/*Vital*Signs*.xls*") + glob.glob("*Vital*Signs*.xls*")
    XLSX = hits[0] if hits else None
YEARS      = list(range(2010, 2024))
HEADER_ROW = 1            # row 0 of each raw sheet is a column-number ruler (original, section 0)

for p in (TABLEAU, XLSX):
    print(f"{'found' if p and os.path.exists(p) else 'MISSING':8s} {p}")
if not os.path.exists(TABLEAU):
    raise FileNotFoundError("Run Vital_Signs_Cleaning_and_EDA.ipynb first -- this notebook "
                            "reads its Tableau workbook rather than re-cleaning.")

log = []
def note(msg):
    """Record an assumption or decision -- same convention as the original notebook."""
    log.append(msg); print("  NOTE:", msg)

def norm_name(c):
    """Copied verbatim from the original notebook (section 1) so header matching is identical."""
    return re.sub(r"\s+", " ", str(c)).strip()

hist    = pd.read_excel(TABLEAU, sheet_name="Historical_Trends")
profile = pd.read_excel(TABLEAU, sheet_name="Service_Need_Profile")
hist["Indicator"] = hist["Indicator"].map(norm_name)   # already normalised; harmless re-apply

print(f"\nHistorical_Trends:    {len(hist):,} rows, {hist.Indicator.nunique()} indicators, "
      f"{hist.CSA.nunique()} CSAs, years {hist.Year.min()}-{hist.Year.max()}")
print(f"Service_Need_Profile: {len(profile)} CSAs x {profile.shape[1]} cols")

# Rebuild the original's frames-by-year structure from the long sheet rather than from raw:
# one wide CSA x indicator frame per year.
frames = {y: g.pivot_table(index="CSA", columns="Indicator", values="Value", aggfunc="first")
          for y, g in hist.groupby("Year")}
city   = (hist.dropna(subset=["CityValue"])
          .drop_duplicates(["Year", "Indicator"])
          .set_index(["Year", "Indicator"]).CityValue)
print(f"frames: {len(frames)} years, "
      f"{sorted({f.shape[0] for f in frames.values()})} CSAs per year")
note("Values come from Historical_Trends, which drops null rows -- so a CSA missing from a "
     "year's frame means the original coded it '--' (suppressed/not collected), not zero.")

found    tableau/Baltimore_Vital_Signs_Tableau.xlsx
found    raw/Vital Signs 2010-2023.xlsx



Historical_Trends:    95,437 rows, 191 indicators, 55 CSAs, years 2010-2023
Service_Need_Profile: 55 CSAs x 13 cols


frames: 14 years, [55] CSAs per year
  NOTE: Values come from Historical_Trends, which drops null rows -- so a CSA missing from a year's frame means the original coded it '--' (suppressed/not collected), not zero.


## 1. Identify the indicator

Search every raw sheet's header row (normalised with `norm_name`) for prenatal / first-trimester
and child-lead fields, then check each hit against `Historical_Trends` for coverage.

In [2]:
PAT_PRENATAL = re.compile(r"prenatal|first trimester", re.I)
PAT_LEAD     = re.compile(r"\blead\b", re.I)        # word boundary: not "leading", "mislead"

raw_hits = {}
if XLSX and os.path.exists(XLSX):
    for y in YEARS:
        hdr = pd.read_excel(XLSX, sheet_name=str(y), header=HEADER_ROW, nrows=0).columns
        for c in (norm_name(c) for c in hdr if not str(c).startswith("Unnamed")):
            if PAT_PRENATAL.search(c) or PAT_LEAD.search(c):
                raw_hits.setdefault(c, []).append(y)
else:
    note("Raw workbook not found -- falling back to Historical_Trends names only.")
    for c in hist.Indicator.unique():
        if PAT_PRENATAL.search(c) or PAT_LEAD.search(c):
            raw_hits[c] = sorted(hist.loc[hist.Indicator == c, "Year"].unique())

def populated(ind, min_n=40):
    """Years in which >= min_n of 55 CSAs carry a value -- the original's usability bar."""
    n = hist[hist.Indicator == ind].groupby("Year").CSA.nunique()
    return [int(y) for y, k in n.items() if k >= min_n], n

print("Header matches across the 14 raw year-sheets:\n")
for c, ys in raw_hits.items():
    ok, n = populated(c)
    kind = "PRENATAL" if PAT_PRENATAL.search(c) else "LEAD"
    print(f"  [{kind}] {c}")
    print(f"      column present in sheets: {ys[0]}-{ys[-1]} ({len(ys)} sheets)")
    print(f"      populated (>=40 CSAs):    {ok if ok else 'never -- empty in every year'}")
    if ok:
        print(f"      CSAs with a value, by year: {dict(n.astype(int))}")
    print()

prenatal_cols = [c for c in raw_hits if PAT_PRENATAL.search(c)]
prenatal_live = [c for c in prenatal_cols if populated(c)[0]]
if len(prenatal_live) != 1:
    raise ValueError(f"Expected exactly one populated prenatal measure, found {prenatal_live}")
PN = prenatal_live[0]
PN_YEARS = populated(PN)[0]

print(f">>> One unambiguous early-prenatal-care measure:\n    '{PN}'")
print(f">>> Available {PN_YEARS[0]}-{PN_YEARS[-1]} ({len(PN_YEARS)} years), "
      f"all 55 CSAs in every one of those years.")
print(f">>> Not collected {PN_YEARS[-1]+1}-{YEARS[-1]}: there is no post-{PN_YEARS[-1]} "
      f"prenatal figure to compare against current (2023) conditions.")
note(f"Prenatal indicator = '{PN}' -- matched by header text; no alternate spelling exists "
     f"across the {len(YEARS)} sheets, so norm_name had nothing to merge here.")

lead_cols = [c for c in raw_hits if PAT_LEAD.search(c)]
print(f"\nSecondary candidate -- child lead ({len(lead_cols)} fields, NOT analysed here):")
for c in lead_cols:
    ok, n = populated(c)
    unit = "count" if c.lower().startswith("number") else "percent"
    if ok:
        print(f"  - {c}  [{unit}]  {ok[0]}-{ok[-1]}, {int(n.min())}-{int(n.max())} CSAs/yr")
    else:
        print(f"  - {c}  [{unit}]  header present, EMPTY in every year -- unusable")
note("The findings doc's 'child lead screening' is the COUNT of children 0-6 tested -- a "
     "count, so it is not comparable across CSAs without a child-population denominator. "
     "The percent-elevated field measures exposure, not access. Both have gaps from 2013 "
     "(47-52 CSAs). Flagged for a later pass, not used below.")

Header matches across the 14 raw year-sheets:

  [PRENATAL] Percent of Births Where the Mother Received Early Prenatal Care (First Trimester)
      column present in sheets: 2010-2023 (14 sheets)
      populated (>=40 CSAs):    [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
      CSAs with a value, by year: {2010: np.int64(55), 2011: np.int64(55), 2012: np.int64(55), 2013: np.int64(55), 2014: np.int64(55), 2015: np.int64(55), 2016: np.int64(55), 2017: np.int64(55), 2018: np.int64(55), 2019: np.int64(55)}

  [LEAD] Number of Children (aged 0-6) Tested for Elevated Blood Lead Levels
      column present in sheets: 2010-2023 (14 sheets)
      populated (>=40 CSAs):    [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
      CSAs with a value, by year: {2010: np.int64(55), 2011: np.int64(55), 2012: np.int64(55), 2013: np.int64(48), 2014: np.int64(52), 2015: np.int64(47), 2016: np.int64(49), 2017: np.int64(50), 2018: np.int64(52), 2019: np.int64(52)}

  [LEAD] Percen

## 2. Headline utilisation gap

For every year the measure exists: city median, min, max, spread, and communities below 60%.
Then check the numbers the findings doc quotes for 2019.

In [3]:
pn_long = hist[hist.Indicator == PN].dropna(subset=["Value"])
pn = pn_long.pivot(index="CSA", columns="Year", values="Value")[PN_YEARS]   # 55 x 10

THRESH = 60
gap = pd.DataFrame({
    "median":   pn.median(),
    "min":      pn.min(),
    "max":      pn.max(),
    "spread":   pn.max() - pn.min(),
    f"n_below_{THRESH}": (pn < THRESH).sum(),
    "worst_CSA": pn.idxmin(),
    "best_CSA":  pn.idxmax(),
    "city_value": [city.get((y, PN), np.nan) for y in PN_YEARS],
}).round(1)
gap.index.name = "Year"
display(gap)

y = PN_YEARS[-1]
g = gap.loc[y]
miss = 100 - g["min"]
print(f"\n{y} headline:")
print(f"  city median {g['median']:.1f}%, range {g['min']:.1f}% ({g.worst_CSA}) to "
      f"{g['max']:.1f}% ({g.best_CSA})")
print(f"  spread = {g.spread:.1f} points; {int(g[f'n_below_{THRESH}'])} of {len(pn)} "
      f"communities below {THRESH}%")
print(f"  in the worst community, {miss:.1f}% of mothers did NOT get first-trimester care")

# Existing findings (Service_Need_Findings.md): 2019 spread 50.6 pts, worst 42.9% (57% miss),
# 21 of 55 below 60%, median 62.5%.
checks = [("spread ~50 pts", abs(g.spread - 50.6) < 0.1),
          ("worst-community miss rate ~57%", abs(miss - 57.1) < 0.2),
          ("21 of 55 below 60%", int(g[f"n_below_{THRESH}"]) == 21),
          ("median 62.5%", abs(g["median"] - 62.5) < 0.1)]
print("\nReplication of existing 2019 findings:")
for label, ok in checks:
    print(f"  [{'OK' if ok else 'MISMATCH'}] {label}")

,median,min,max,spread,n_below_60,worst_CSA,best_CSA,city_value
Year,,,,,,,,
2010,57.2,40.4,74.4,33.9,32,Orangeville/East Highlandtown,Inner Harbor/Federal Hill,57.0
2011,58.9,42.9,79.4,36.5,31,Forest Park/Walbrook,Canton,59.0
2012,63.2,48.1,86.9,38.8,18,Southern Park Heights,Greater Roland Park/Poplar Hill,62.7
2013,47.3,36.7,71.4,34.8,43,Westport/Mount Winans/Lakeland,South Baltimore,49.5
2014,46.6,35.0,73.8,38.9,48,Sandtown-Winchester/Harlem Park,South Baltimore,48.5
2015,50.0,33.6,81.6,47.9,47,Southeastern,Canton,50.4
2016,48.9,37.4,79.2,41.8,44,Pimlico/Arlington/Hilltop,Canton,50.9
2017,63.2,50.9,82.9,32.0,19,Southwest Baltimore,Canton,63.4
2018,63.6,47.7,87.1,39.4,18,Penn North/Reservoir Hill,Canton,64.5



2019 headline:
  city median 62.5%, range 42.9% (Orangeville/East Highlandtown) to 93.5% (Greater Roland Park/Poplar Hill)
  spread = 50.6 points; 21 of 55 communities below 60%
  in the worst community, 57.1% of mothers did NOT get first-trimester care

Replication of existing 2019 findings:
  [OK] spread ~50 pts
  [OK] worst-community miss rate ~57%
  [OK] 21 of 55 below 60%
  [OK] median 62.5%


## 3. Data-quality check: the 2013–2016 break

The findings doc flags a mid-series drop. Rather than assume which years are affected, let the
data find it: sort the ten yearly medians and split at the largest gap. A genuine definitional
or collection change should also leave the **rank order** of communities roughly intact
(everyone shifts together), which a real neighbourhood health event would not.

In [4]:
med = gap["median"]
s = med.sort_values()
cut = s.diff().idxmax()                  # first year above the largest jump between medians
low_cluster = sorted(s[s < s[cut]].index)
hi_cluster  = sorted(s[s >= s[cut]].index)
jump = s[cut] - s[s < s[cut]].max()

print("Citywide median by year (and the city's own aggregate rate):")
for yy in PN_YEARS:
    bar = "#" * int(round(med[yy] - 30))
    flag = "  <-- low cluster" if yy in low_cluster else ""
    print(f"  {yy}  median {med[yy]:5.1f}  city {gap.city_value[yy]:5.1f}  "
          f"below60 {int(gap.loc[yy, 'n_below_60']):>2}  {bar}{flag}")

print(f"\nLargest gap between sorted medians: {jump:.1f} pts. Every other adjacent gap: "
      f"<= {s.diff().drop(cut).max():.1f} pts.")
contiguous = low_cluster == list(range(low_cluster[0], low_cluster[-1] + 1))
bounded = min(PN_YEARS) < low_cluster[0] and max(PN_YEARS) > low_cluster[-1]
print(f"Low cluster: {low_cluster}  contiguous={contiguous}  "
      f"stable years on both sides={bounded}")

# Rank stability across adjacent years. If the drop were a real, uneven health shock the
# cross-over years would reshuffle communities; a uniform measurement change would not.
rk = pn.rank()
adj = pd.Series({f"{a}->{b}": rk[a].corr(rk[b]) for a, b in zip(PN_YEARS, PN_YEARS[1:])})
print("\nSpearman rank correlation, year to adjacent year:")
for k, v in adj.items():
    a, b = map(int, k.split("->"))
    edge = " <-- crosses the break" if (a in low_cluster) != (b in low_cluster) else ""
    print(f"  {k}  {v:.2f}{edge}")

cross = [v for k, v in adj.items()
         if (int(k[:4]) in low_cluster) != (int(k[-4:]) in low_cluster)]
within = adj.drop([k for k in adj.index
                   if (int(k[:4]) in low_cluster) != (int(k[-4:]) in low_cluster)])
print(f"Across the break: {min(cross):.2f}-{max(cross):.2f}; all other adjacent pairs: "
      f"{within.min():.2f}-{within.max():.2f}. Communities keep their relative order "
      f"through the drop and recovery as well as they do in any ordinary year -- the whole "
      f"city shifted together, which is the signature of a measurement change.")

BREAK_YEARS  = low_cluster if (contiguous and bounded) else []
STABLE_YEARS = [yy for yy in PN_YEARS if yy not in BREAK_YEARS]
LATEST       = max(STABLE_YEARS)

print(f"\n>>> BREAK years (do not compare levels with other years): {BREAK_YEARS}")
print(f">>> STABLE years (levels comparable):                     {STABLE_YEARS}")
print(f">>> Latest reliable year for the ranked geography:         {LATEST}")
note(f"Break detected from the data, not hard-coded: medians for {BREAK_YEARS} form a "
     f"separate cluster {jump:.1f} pts below the rest, drop and recover in consecutive "
     f"years, and the citywide aggregate shows the same step. Treated as a collection/"
     f"definitional change -- LEVELS in those years are not compared with any other year.")
note("Within-year comparisons (rankings, correlations) are less exposed to a uniform level "
     "shift, so they are still computed for break years in section 7 but labelled.")
if med[STABLE_YEARS[0]] < med[LATEST] - 4:
    note(f"Even within the stable set, {STABLE_YEARS[0]}-{STABLE_YEARS[1]} medians "
         f"({med[STABLE_YEARS[0]]:.1f}, {med[STABLE_YEARS[1]]:.1f}) sit ~4-6 pts below "
         f"2012/2017-2019. This is within the plausible range of real change, so they are "
         f"kept, but 'below 60%' counts for 2010-2011 are inflated by it (32, 31 vs ~19).")

Citywide median by year (and the city's own aggregate rate):
  2010  median  57.2  city  57.0  below60 32  ###########################
  2011  median  58.9  city  59.0  below60 31  #############################
  2012  median  63.2  city  62.7  below60 18  #################################
  2013  median  47.3  city  49.5  below60 43  #################  <-- low cluster
  2014  median  46.6  city  48.5  below60 48  #################  <-- low cluster
  2015  median  50.0  city  50.4  below60 47  ####################  <-- low cluster
  2016  median  48.9  city  50.9  below60 44  ###################  <-- low cluster
  2017  median  63.2  city  63.4  below60 19  #################################
  2018  median  63.6  city  64.5  below60 18  ##################################
  2019  median  62.5  city  63.1  below60 21  ################################

Largest gap between sorted medians: 7.2 pts. Every other adjacent gap: <= 3.6 pts.
Low cluster: [2013, 2014, 2015, 2016]  contiguous=True  

## 4. Ranked geography — all 55 communities, worst to best

Latest reliable year. Single-year birth-based rates in small CSAs are noisy, so a 3-year mean
over the most recent stable years is shown alongside as a robustness check on the ranking.

In [5]:
recent = [yy for yy in STABLE_YEARS if yy > max(BREAK_YEARS or [0])][-3:]
city_latest = city.get((LATEST, PN), np.nan)

ranked = (pd.DataFrame({
            f"prenatal_{LATEST}": pn[LATEST],
            f"mean_{recent[0]}_{recent[-1]}": pn[recent].mean(axis=1),
          })
          .assign(vs_city=lambda d: d[f"prenatal_{LATEST}"] - city_latest,
                  below_60=lambda d: d[f"prenatal_{LATEST}"] < THRESH)
          # worst first; CSA name as a deterministic tiebreak, as in the original
          .reset_index().sort_values([f"prenatal_{LATEST}", "CSA"], kind="mergesort")
          .reset_index(drop=True))
ranked.insert(0, "rank_worst_first", range(1, len(ranked) + 1))
ranked = ranked.round(1)

print(f"Early prenatal care, {LATEST}: all {len(ranked)} CSAs, worst (1) to best "
      f"({len(ranked)}). City aggregate = {city_latest:.1f}%.")
display(ranked)

r3 = ranked[f"prenatal_{LATEST}"].rank().corr(ranked[f"mean_{recent[0]}_{recent[-1]}"].rank())
top10_single = set(ranked.head(10).CSA)
top10_mean = set(ranked.nsmallest(10, f"mean_{recent[0]}_{recent[-1]}").CSA)
print(f"\nRank agreement, {LATEST} alone vs {recent[0]}-{recent[-1]} mean: rho = {r3:.2f}; "
      f"{len(top10_single & top10_mean)} of the worst 10 are the same under both.")
print(f"{int(ranked.below_60.sum())} CSAs below {THRESH}%; "
      f"{int((ranked.vs_city < 0).sum())} below the city aggregate.")

Early prenatal care, 2019: all 55 CSAs, worst (1) to best (55). City aggregate = 63.1%.


,rank_worst_first,CSA,prenatal_2019,mean_2017_2019,vs_city,below_60
0,1,Orangeville/East Highlandtown,42.9,53.8,-20.2,True
1,2,Madison/East End,44.7,56.3,-18.4,True
2,3,Cross-Country/Cheswolde,47.8,54.6,-15.3,True
3,4,Poppleton/The Terraces/Hollins Market,49.3,59.9,-13.8,True
4,5,Westport/Mount Winans/Lakeland,50.8,52.4,-12.2,True
5,6,Brooklyn/Curtis Bay/Hawkins Point,51.5,57.3,-11.6,True
6,7,Glen-Fallstaff,52.1,51.5,-11.0,True
7,8,Pimlico/Arlington/Hilltop,53.6,59.7,-9.5,True
8,9,Edmondson Village,53.9,55.5,-9.1,True
9,10,Morrell Park/Violetville,54.1,57.0,-9.0,True



Rank agreement, 2019 alone vs 2017-2019 mean: rho = 0.91; 7 of the worst 10 are the same under both.
21 CSAs below 60%; 29 below the city aggregate.


## 5. Overlap with the existing need tiers

`Tier` comes straight from `Service_Need_Profile` (the four-dimension Composite, cut at
40 / 55 / 70). The overlap is only meaningful against the **base rate** — the share of all 55
communities that are Highest/High anyway — so both are reported.

In [6]:
HIGH_TIERS = ["Highest", "High"]
tiers = profile.set_index("CSA")[["Tier", "Composite"]]

missing = sorted(set(ranked.CSA) ^ set(tiers.index))
if missing:
    note(f"CSA names not matching between prenatal ranking and profile: {missing}")
else:
    print(f"All {len(ranked)} CSAs join cleanly to Service_Need_Profile on CSA.\n")

rt = ranked.merge(tiers, left_on="CSA", right_index=True, how="left")
base = rt.Tier.isin(HIGH_TIERS).mean()
print(f"Base rate: {rt.Tier.isin(HIGH_TIERS).sum()} of {len(rt)} CSAs "
      f"({base:.0%}) are Highest/High tier.\n")

print(f"{'worst N':>8} {'in Highest/High':>16} {'share':>6} {'lift vs base':>13}")
for N in (10, 15):
    w = rt.head(N)
    k = int(w.Tier.isin(HIGH_TIERS).sum())
    print(f"{N:>8} {f'{k} of {N}':>16} {k/N:>6.0%} {k/N/base:>12.2f}x")

print("\nTier mix of the worst 15 vs all 55:")
mix = pd.DataFrame({"worst_15": rt.head(15).Tier.value_counts(),
                    "all_55": rt.Tier.value_counts()}).reindex(
      ["Highest", "High", "Moderate", "Lower"]).fillna(0).astype(int)
display(mix)

print("Worst 15 for prenatal care, with need tier:")
display(rt.head(15)[["rank_worst_first", "CSA", f"prenatal_{LATEST}", "Tier", "Composite"]])

out = rt.head(15)[~rt.head(15).Tier.isin(HIGH_TIERS)]
print(f"Worst-15 prenatal CSAs NOT in a high tier ({len(out)}): "
      + (", ".join(f"{r.CSA} [{r.Tier}]" for _, r in out.iterrows()) or "none"))
hi_ok = rt[rt.Tier.eq("Highest") & (rt.rank_worst_first > 27)]
print(f"Highest-tier CSAs in the BETTER half for prenatal care ({len(hi_ok)}): "
      + (", ".join(hi_ok.CSA) or "none"))
print(f"\nRank correlation, need Composite vs prenatal-care shortfall: "
      f"rho = {rt.Composite.rank().corr((-rt[f'prenatal_{LATEST}']).rank()):.2f}")

note(f"Vintage mismatch: Tier is built from {2023} indicators; prenatal care is {LATEST}. "
     f"The original notebook warns against combining vintages -- this overlap compares a "
     f"{LATEST} outcome with 2023 need, so read it as 'do today's high-need areas include "
     f"the places that were missing prenatal care', not as a same-year relationship.")

All 55 CSAs join cleanly to Service_Need_Profile on CSA.

Base rate: 28 of 55 CSAs (51%) are Highest/High tier.

 worst N  in Highest/High  share  lift vs base
      10          8 of 10    80%         1.57x
      15         12 of 15    80%         1.57x

Tier mix of the worst 15 vs all 55:


,worst_15,all_55
Tier,,
Highest,6,13
High,6,15
Moderate,2,9
Lower,1,18


Worst 15 for prenatal care, with need tier:


,rank_worst_first,CSA,prenatal_2019,Tier,Composite
0,1,Orangeville/East Highlandtown,42.9,Moderate,53.0
1,2,Madison/East End,44.7,High,68.6
2,3,Cross-Country/Cheswolde,47.8,Lower,20.4
3,4,Poppleton/The Terraces/Hollins Market,49.3,High,69.6
4,5,Westport/Mount Winans/Lakeland,50.8,High,65.0
5,6,Brooklyn/Curtis Bay/Hawkins Point,51.5,Highest,70.2
6,7,Glen-Fallstaff,52.1,High,57.0
7,8,Pimlico/Arlington/Hilltop,53.6,Highest,78.6
8,9,Edmondson Village,53.9,High,59.8
9,10,Morrell Park/Violetville,54.1,Highest,71.1


Worst-15 prenatal CSAs NOT in a high tier (3): Orangeville/East Highlandtown [Moderate], Cross-Country/Cheswolde [Lower], Forest Park/Walbrook [Moderate]
Highest-tier CSAs in the BETTER half for prenatal care (1): Oldtown/Middle East

Rank correlation, need Composite vs prenatal-care shortfall: rho = 0.62
  NOTE: Vintage mismatch: Tier is built from 2023 indicators; prenatal care is 2019. The original notebook warns against combining vintages -- this overlap compares a 2019 outcome with 2023 need, so read it as 'do today's high-need areas include the places that were missing prenatal care', not as a same-year relationship.


## 6. Persistence, stable years only

Mirrors the original's life-expectancy logic (section 14): for each year, take the 10 lowest
values (CSA name as a deterministic tiebreak) and count how often each community appears.
Break years are excluded, so the maximum possible count is the number of stable years.

In [7]:
WORST_N = 10
pcount = pd.Series(0, index=pn.index, dtype="int64")
for yy in STABLE_YEARS:
    worst = (pn_long[pn_long.Year == yy]
             .sort_values(["Value", "CSA"], kind="mergesort")
             .CSA.head(WORST_N))
    pcount[pcount.index.intersection(worst)] += 1

total = int(pcount.sum())
print(f"Worst-{WORST_N} appearances over {len(STABLE_YEARS)} stable years {STABLE_YEARS}; "
      f"total flagged = {total} (expected {len(STABLE_YEARS) * WORST_N})\n")

dist = pcount.value_counts().sort_index(ascending=False)
print("How many CSAs at each persistence count:")
for k, n in dist.items():
    print(f"  {k} of {len(STABLE_YEARS)}: {n:>2} CSAs")

pers = (pd.DataFrame({"persistence_count": pcount})
        .join(pn[STABLE_YEARS].round(1))
        .join(tiers.Tier)
        .sort_values(["persistence_count", LATEST], ascending=[False, True]))
PERSIST_MIN = len(STABLE_YEARS) // 2          # worst-10 in at least half the stable years
chronic = pers[pers.persistence_count >= PERSIST_MIN]
print(f"\nHighest persistence -- worst {WORST_N} in at least {PERSIST_MIN} of "
      f"{len(STABLE_YEARS)} stable years ({len(chronic)} CSAs):")
display(chronic)

never, ever = int((pcount == 0).sum()), int((pcount >= 1).sum())
print(f"{never} of {len(pcount)} CSAs never appear in a stable-year worst {WORST_N}; "
      f"{ever} appear at least once (the minimum possible is {WORST_N}).")
print(f"Only {int((pcount == len(STABLE_YEARS)).sum())} CSA is in the worst {WORST_N} every "
      f"stable year: {', '.join(pcount[pcount == len(STABLE_YEARS)].index)}.")
le8 = profile[profile.Persistence_Count == profile.Persistence_Count.max()]
print(f"For comparison, life expectancy (original notebook): {len(le8)} CSAs sit in the worst "
      f"10 in all {int(profile.Persistence_Count.max())} years. The prenatal worst-10 turns "
      f"over far more -- expected for a birth-based rate with small annual denominators -- so "
      f"a single year's ranking should not be read as a fixed list.")
print(f"High-persistence CSAs in Highest/High tier: "
      f"{int(chronic.Tier.isin(HIGH_TIERS).sum())} of {len(chronic)}")

# Does persistence depend on the break-year exclusion? Show counts using the break years.
brk = pd.Series(0, index=pn.index, dtype="int64")
for yy in BREAK_YEARS:
    worst = pn_long[pn_long.Year == yy].sort_values(["Value", "CSA"], kind="mergesort").CSA.head(WORST_N)
    brk[brk.index.intersection(worst)] += 1
print(f"\nSame high-persistence CSAs during the break years {BREAK_YEARS} (worst-{WORST_N} count):")
for c in chronic.index:
    print(f"  {c:40s} {brk[c]} of {len(BREAK_YEARS)}")
note(f"Persistence_Count in Service_Need_Profile is LIFE EXPECTANCY (2011-2018, 8 years). "
     f"The prenatal count here is a different measure over {len(STABLE_YEARS)} stable years. "
     f"The export names it persistence_count as requested -- rename it in Tableau if both "
     f"sources are joined, or the two will be confused.")

Worst-10 appearances over 6 stable years [2010, 2011, 2012, 2017, 2018, 2019]; total flagged = 60 (expected 60)

How many CSAs at each persistence count:
  6 of 6:  1 CSAs
  4 of 6:  2 CSAs
  3 of 6:  3 CSAs
  2 of 6: 11 CSAs
  1 of 6: 15 CSAs
  0 of 6: 23 CSAs

Highest persistence -- worst 10 in at least 3 of 6 stable years (6 CSAs):


,persistence_count,2010,2011,2012,2017,2018,2019,Tier
CSA,,,,,,,,
Westport/Mount Winans/Lakeland,6,47.1,51.8,57.1,51.6,54.8,50.8,High
Orangeville/East Highlandtown,4,40.4,45.1,58.3,52.8,65.6,42.9,Moderate
Southwest Baltimore,4,50.0,45.7,51.2,50.9,60.7,55.5,Highest
Brooklyn/Curtis Bay/Hawkins Point,3,50.9,47.2,49.6,61.2,59.1,51.5,Highest
Glen-Fallstaff,3,56.4,60.6,62.6,54.5,48.0,52.1,High
Midway/Coldstream,3,45.2,53.3,51.4,56.6,58.5,55.4,Highest


23 of 55 CSAs never appear in a stable-year worst 10; 32 appear at least once (the minimum possible is 10).
Only 1 CSA is in the worst 10 every stable year: Westport/Mount Winans/Lakeland.
For comparison, life expectancy (original notebook): 7 CSAs sit in the worst 10 in all 8 years. The prenatal worst-10 turns over far more -- expected for a birth-based rate with small annual denominators -- so a single year's ranking should not be read as a fixed list.
High-persistence CSAs in Highest/High tier: 5 of 6

Same high-persistence CSAs during the break years [2013, 2014, 2015, 2016] (worst-10 count):
  Westport/Mount Winans/Lakeland           3 of 4
  Orangeville/East Highlandtown            1 of 4
  Southwest Baltimore                      3 of 4
  Brooklyn/Curtis Bay/Hawkins Point        1 of 4
  Glen-Fallstaff                           0 of 4
  Midway/Coldstream                        1 of 4
  NOTE: Persistence_Count in Service_Need_Profile is LIFE EXPECTANCY (2011-2018, 8 years). The p

## 7. Income as the driver

**Income field.** `Median Household Income` — the income field the original notebook carries in
`Service_Need_Profile` (via `CONTEXT`), taken here from `Historical_Trends` for the **same year**
as the prenatal value, not the 2023 profile figure.

**Method: income-residualised (first-order partial) Pearson correlation.** Regress both the
barrier and prenatal care on income by OLS, then correlate the residuals. This is exactly the
partial correlation r(barrier, prenatal | income). Chosen because:
1. it is what the existing findings used — it reproduces them to two decimals, so the numbers
   are directly comparable;
2. with n = 55 and one control, a linear adjustment is about all the data can support;
3. residualising makes the "after adjusting" number concrete: *among communities of similar
   income, does more of this barrier go with less early care?*

Income is skewed, so a log-income version is shown alongside as a sensitivity check.
No p-values: the 55 CSAs are the whole population of interest, not a sample, and the
original notebook reports none either.

In [8]:
INC = "Median Household Income"
BARRIERS = {                                  # names as in the original's NEED / DIMENSIONS lists
    "No internet at home":     "Percent of Households with No Internet at Home",
    "No HS diploma":           "Percent Population (25 Years and over) With Less Than a High School Diploma or GED",
    "No vehicle":              "Percent of Households with No Vehicles Available",
    "Public transport to work": "Percent of Population that Uses Public Transportation to Get to Work",
}
for c in [INC, *BARRIERS.values()]:
    ys = sorted(hist.loc[hist.Indicator == c, "Year"].unique())
    print(f"  {c[:70]:70s} {ys[0]}-{ys[-1]}")

def resid(v, z):
    X = np.column_stack([np.ones(len(z)), z])
    return v - X @ np.linalg.lstsq(X, v, rcond=None)[0]

def partial_r(x, y, z):
    return float(np.corrcoef(resid(x, z), resid(y, z))[0, 1])

# --- 7a. income vs prenatal care, every year
inc_rows = []
for yy in PN_YEARS:
    d = frames[yy][[INC, PN]].dropna()
    inc_rows.append({"Year": yy, "n": len(d),
                     "r_income": d[INC].corr(d[PN]),
                     "r_log_income": np.log(d[INC]).corr(d[PN]),
                     "rho_spearman": d[INC].rank().corr(d[PN].rank()),
                     "break_year": yy in BREAK_YEARS})
inc_tbl = pd.DataFrame(inc_rows).set_index("Year").round(3)
print("\n7a. Median household income vs early prenatal care, by year:")
display(inc_tbl)
st = inc_tbl.loc[STABLE_YEARS, "r_income"]
print(f"Stable years: r ranges {st.min():+.2f} to {st.max():+.2f} (mean {st.mean():+.2f}). "
      f"Break years: {inc_tbl.loc[BREAK_YEARS, 'r_income'].min():+.2f} to "
      f"{inc_tbl.loc[BREAK_YEARS, 'r_income'].max():+.2f} -- the relationship survives "
      f"the level shift, consistent with a measurement change rather than a real reshuffle.")

# --- 7b. each barrier, raw and income-adjusted, every year both exist
rows = []
for yy in PN_YEARS:
    f = frames[yy]
    for lab, col in BARRIERS.items():
        if col not in f.columns:
            continue
        d = f[[col, PN, INC]].dropna()
        if len(d) < 40:
            continue
        x, y_, z = d[col].values, d[PN].values, d[INC].values
        rows.append({"Year": yy, "Barrier": lab, "n": len(d),
                     "r_alone": np.corrcoef(x, y_)[0, 1],
                     "r_adj_income": partial_r(x, y_, z),
                     "r_adj_log_income": partial_r(x, y_, np.log(z)),
                     "r_barrier_income": np.corrcoef(x, z)[0, 1],
                     "break_year": yy in BREAK_YEARS})
bar = pd.DataFrame(rows)
bar["weakened"] = bar.r_adj_income.abs() < bar.r_alone.abs()
bar["reversed"] = np.sign(bar.r_adj_income) != np.sign(bar.r_alone)
bar["weakened_or_reversed"] = bar.weakened | bar.reversed

print(f"\n7b. Barrier vs early prenatal care, {LATEST} (negative = more barrier, less early care):")
hl = bar[bar.Year == LATEST].set_index("Barrier")
display(hl[["r_alone", "r_adj_income", "r_adj_log_income", "r_barrier_income",
            "weakened", "reversed"]].round(3))

EXISTING = {"No internet at home": (-0.66, -0.16), "No HS diploma": (-0.70, -0.36),
            "No vehicle": (-0.48, 0.29), "Public transport to work": (-0.51, 0.14)}
print(f"Replication of existing {LATEST} findings (alone -> adjusted):")
for lab, (a, b) in EXISTING.items():
    r = hl.loc[lab]
    ok = abs(r.r_alone - a) < 0.006 and abs(r.r_adj_income - b) < 0.006
    print(f"  [{'OK' if ok else 'MISMATCH'}] {lab:26s} expected {a:+.2f} -> {b:+.2f}   "
          f"got {r.r_alone:+.2f} -> {r.r_adj_income:+.2f}")
print(f"  [{'OK' if abs(inc_tbl.loc[LATEST, 'r_income'] - 0.74) < 0.006 else 'MISMATCH'}] "
      f"income alone expected +0.74, got {inc_tbl.loc[LATEST, 'r_income']:+.2f}")

print("\nDoes the pattern hold beyond one year? Stable years only:")
sb = bar[~bar.break_year]
summ = sb.groupby("Barrier").agg(
    years=("Year", lambda s: f"{s.min()}-{s.max()} ({len(s)})"),
    r_alone_mean=("r_alone", "mean"), r_adj_mean=("r_adj_income", "mean"),
    weakened=("weakened", "sum"), reversed=("reversed", "sum"),
    weakened_or_reversed=("weakened_or_reversed", "sum")).round(2)
display(summ)

print("Full barrier x year table (break years labelled):")
display(bar.pivot(index="Barrier", columns="Year", values="r_adj_income")
        .rename(columns=lambda c: f"{c}*" if c in BREAK_YEARS else c).round(2))
print("  values = income-adjusted r;  * = break year")

for lab, g in sb.groupby("Barrier"):
    w, r_, n_ = int(g.weakened.sum()), int(g.reversed.sum()), len(g)
    print(f"  {lab:26s} weakened in {w}/{n_} stable years, sign reversed in {r_}/{n_}")

k, n_ = int(sb.weakened_or_reversed.sum()), len(sb)
print(f"\n>>> CONFIRMED: in {k} of {n_} barrier-year pairs (stable years), the barrier's "
      f"association with early care weakens or reverses once income is held fixed.")
print(f">>> Income alone (r = {inc_tbl.loc[LATEST, 'r_income']:+.2f} in {LATEST}) is stronger "
      f"than any barrier alone ({hl.r_alone.abs().max():.2f} at most), and every barrier "
      f"correlates {hl.r_barrier_income.abs().min():.2f}-{hl.r_barrier_income.abs().max():.2f} "
      f"with income itself -- the barriers largely stand in for income.")

same_sign = (np.sign(sb.r_adj_income) == np.sign(sb.r_adj_log_income))
print(f"Log-income sensitivity: adjusted r keeps its sign in {int(same_sign.sum())} of "
      f"{len(sb)} stable barrier-years"
      + ("" if same_sign.all() else " -- the exceptions:"))
if not same_sign.all():
    display(sb[~same_sign][["Year", "Barrier", "r_adj_income", "r_adj_log_income"]].round(3))
note("Partial correlation by OLS residualisation on raw Median Household Income (reproduces "
     "the existing findings); log-income shown as a sensitivity check.")
# --- 7c. where income does NOT explain the gap: actual minus income-predicted, LATEST year
d = frames[LATEST][[INC, PN]].dropna()
beta = np.polyfit(d[INC], d[PN], 1)
d["predicted"] = np.polyval(beta, d[INC])
d["shortfall_vs_income"] = d[PN] - d["predicted"]
d = d.join(profile.set_index("CSA")[["Tier", "Pct Hispanic"]])
print(f"\n7c. {LATEST}: income explains R^2 = {d[INC].corr(d[PN])**2:.2f} of the variation in "
      f"early care across CSAs. Largest shortfalls against what income predicts:")
display(d.nsmallest(8, "shortfall_vs_income")
        [[PN, "predicted", "shortfall_vs_income", INC, "Tier", "Pct Hispanic"]]
        .rename(columns={PN: f"prenatal_{LATEST}", INC: f"income_{LATEST}",
                         "Pct Hispanic": "Pct Hispanic (2023)"}).round(1))
print("These communities miss early care more than their income accounts for, so an income-"
      "only story under-targets them. Several are not high-need tiers. The residual is "
      "a lead for follow-up (language access, local provider supply), not a finding: "
      "this workbook has no variable that explains it.")
note("Pct Hispanic in 7c is the 2023 value carried in Service_Need_Profile, shown as context "
     "only -- the original notebook labels it a rough language-access proxy.")

note("As in the findings doc: adjusting for income may remove the pathway of interest "
     "(poverty -> no car/internet -> missed care) rather than debunk it. These are "
     "ecological correlations across 55 areas and cannot separate the two readings.")

  Median Household Income                                                2010-2023


  Percent of Households with No Internet at Home                         2017-2023
  Percent Population (25 Years and over) With Less Than a High School Di 2011-2023
  Percent of Households with No Vehicles Available                       2011-2023
  Percent of Population that Uses Public Transportation to Get to Work   2011-2023

7a. Median household income vs early prenatal care, by year:


,n,r_income,r_log_income,rho_spearman,break_year
Year,,,,,
2010,55,0.617,0.545,0.564,False
2011,55,0.671,0.626,0.607,False
2012,55,0.688,0.575,0.470,False
2013,55,0.811,0.771,0.719,True
2014,55,0.714,0.647,0.541,True
2015,55,0.797,0.724,0.687,True
2016,55,0.845,0.777,0.696,True
2017,55,0.739,0.672,0.626,False
2018,55,0.792,0.709,0.572,False


Stable years: r ranges +0.62 to +0.79 (mean +0.71). Break years: +0.71 to +0.84 -- the relationship survives the level shift, consistent with a measurement change rather than a real reshuffle.



7b. Barrier vs early prenatal care, 2019 (negative = more barrier, less early care):


...

,Year,Barrier,r_adj_income,r_adj_log_income
21,2017,Public transport to work,0.006,-0.068


  NOTE: Partial correlation by OLS residualisation on raw Median Household Income (reproduces the existing findings); log-income shown as a sensitivity check.



7c. 2019: income explains R^2 = 0.55 of the variation in early care across CSAs. Largest shortfalls against what income predicts:


,prenatal_2019,predicted,shortfall_vs_income,income_2019,Tier,Pct Hispanic (2023)
CSA,,,,,,
Orangeville/East Highlandtown,42.9,66.9,-24.0,60456.8,Moderate,44.4
Cross-Country/Cheswolde,47.8,66.4,-18.6,58843.7,Lower,5.5
Madison/East End,44.7,59.5,-14.8,37328.3,High,6.4
Patterson Park North & East,58.2,71.0,-12.8,73313.3,Lower,11.1
Brooklyn/Curtis Bay/Hawkins Point,51.5,60.1,-8.6,39162.1,Highest,31.4
Glen-Fallstaff,52.1,60.6,-8.5,40970.3,High,8.6
Westport/Mount Winans/Lakeland,50.8,59.0,-8.2,35923.2,High,40.1
Morrell Park/Violetville,54.1,62.0,-7.9,45125.0,Highest,9.5


These communities miss early care more than their income accounts for, so an income-only story under-targets them. Several are not high-need tiers. The residual is a lead for follow-up (language access, local provider supply), not a finding: this workbook has no variable that explains it.
  NOTE: Pct Hispanic in 7c is the 2023 value carried in Service_Need_Profile, shown as context only -- the original notebook labels it a rough language-access proxy.
  NOTE: As in the findings doc: adjusting for income may remove the pathway of interest (poverty -> no car/internet -> missed care) rather than debunk it. These are ecological correlations across 55 areas and cannot separate the two readings.


## 8. Output for Tableau

One row per CSA. Written as a standalone CSV rather than appended to
`Baltimore_Vital_Signs_Tableau.xlsx`, because the original notebook rewrites that workbook from
scratch when re-run and would silently drop any extra sheet. Join on `CSA` — it is the same
`CSA2010` key used by every sheet in the main workbook.

In [9]:
export = (ranked[["CSA", f"prenatal_{LATEST}", "rank_worst_first"]]
          .rename(columns={f"prenatal_{LATEST}": "prenatal_care_pct",
                           "rank_worst_first": "prenatal_rank"})
          .merge(tiers.Tier.rename("need_tier"), left_on="CSA", right_index=True, how="left")
          .merge(pcount.rename("persistence_count"), left_on="CSA", right_index=True, how="left")
          .merge(frames[LATEST][INC].round(0).rename("income"),
                 left_on="CSA", right_index=True, how="left"))
export = export[["CSA", "prenatal_care_pct", "need_tier", "persistence_count", "income",
                 "prenatal_rank"]]

assert len(export) == 55 and export.CSA.is_unique
assert export.drop(columns="prenatal_rank").notna().all().all(), "unexpected nulls in export"

CSV_OUT = f"{OUT}/Prenatal_Care_Deep_Dive.csv"
export.to_csv(CSV_OUT, index=False)

print(f"{CSV_OUT}: {export.shape[0]} rows x {export.shape[1]} cols\n")
print("Column definitions:")
print(f"  CSA                join key, = CSA2010 / 'CSA' in every sheet of the main workbook")
print(f"  prenatal_care_pct  % births with first-trimester care, {LATEST} (latest reliable year)")
print(f"  need_tier          Tier from Service_Need_Profile (2023 four-dimension Composite)")
print(f"  persistence_count  times in the worst {WORST_N} for prenatal care, "
      f"0-{len(STABLE_YEARS)}, stable years {STABLE_YEARS}")
print(f"  income             {INC}, {LATEST} dollars, same year as prenatal_care_pct")
print(f"  prenatal_rank      1 = worst, {len(export)} = best (extra: sort key for Tableau)")
display(export.head(15))

tableau/Prenatal_Care_Deep_Dive.csv: 55 rows x 6 cols

Column definitions:
  CSA                join key, = CSA2010 / 'CSA' in every sheet of the main workbook
  prenatal_care_pct  % births with first-trimester care, 2019 (latest reliable year)
  need_tier          Tier from Service_Need_Profile (2023 four-dimension Composite)
  persistence_count  times in the worst 10 for prenatal care, 0-6, stable years [2010, 2011, 2012, 2017, 2018, 2019]
  income             Median Household Income, 2019 dollars, same year as prenatal_care_pct
  prenatal_rank      1 = worst, 55 = best (extra: sort key for Tableau)


,CSA,prenatal_care_pct,need_tier,persistence_count,income,prenatal_rank
0,Orangeville/East Highlandtown,42.9,Moderate,4,60457.0,1
1,Madison/East End,44.7,High,2,37328.0,2
2,Cross-Country/Cheswolde,47.8,Lower,2,58844.0,3
3,Poppleton/The Terraces/Hollins Market,49.3,High,2,23403.0,4
4,Westport/Mount Winans/Lakeland,50.8,High,6,35923.0,5
5,Brooklyn/Curtis Bay/Hawkins Point,51.5,Highest,3,39162.0,6
6,Glen-Fallstaff,52.1,High,3,40970.0,7
7,Pimlico/Arlington/Hilltop,53.6,Highest,2,32883.0,8
8,Edmondson Village,53.9,High,2,43627.0,9
9,Morrell Park/Violetville,54.1,Highest,2,45125.0,10


## Assumptions & decisions log

In [10]:
print(f"{len(log)} notes recorded:\n")
for i, m in enumerate(log, 1):
    print(f"{i:>2}. {m}\n")

11 notes recorded:

 1. Values come from Historical_Trends, which drops null rows -- so a CSA missing from a year's frame means the original coded it '--' (suppressed/not collected), not zero.

 2. Prenatal indicator = 'Percent of Births Where the Mother Received Early Prenatal Care (First Trimester)' -- matched by header text; no alternate spelling exists across the 14 sheets, so norm_name had nothing to merge here.

 3. The findings doc's 'child lead screening' is the COUNT of children 0-6 tested -- a count, so it is not comparable across CSAs without a child-population denominator. The percent-elevated field measures exposure, not access. Both have gaps from 2013 (47-52 CSAs). Flagged for a later pass, not used below.

 4. Break detected from the data, not hard-coded: medians for [2013, 2014, 2015, 2016] form a separate cluster 7.2 pts below the rest, drop and recover in consecutive years, and the citywide aggregate shows the same step. Treated as a collection/definitional change --

---

## What to say from this

1. **The gap is real and large in the reliable years** — see section 2 for the exact
   spread and miss rate, replicated from the existing findings.
2. **Do not claim a trend.** The 2013–2016 values sit in a separate cluster and move together;
   only 2010–2012 and 2017–2019 are comparable to each other.
3. **Income is the organising variable.** Its correlation with early care is strong in every
   year, including across the break; the access barriers lose most of their association once
   income is held fixed, and car ownership reverses.
4. **The overlap with need tiers is quantified against a base rate** — quote the lift, not
   just the count.